In [1]:
# ============================================================
# IV Fluid Level Detector v2 + Gradio
# يحسب النسبة حسب موقع "خط السائل" الفعلي داخل الكيس (أدق بكثير)
# ============================================================

!pip install opencv-python-headless gradio -q

import cv2
import numpy as np
import gradio as gr

# ============================================================
# دالة إيجاد صندوق الكيس (bounding box)
# ============================================================
def find_bag_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

    lower = np.array([0, 0, 150])
    upper = np.array([180, 60, 255])

    mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)

    # تجاهل كونتورات صغيرة جداً (ضوضاء)
    if w * h < (img.shape[0] * img.shape[1] * 0.01):
        return None

    return x, y, w, h


# ============================================================
# دالة إيجاد خط السائل داخل الكيس
# ============================================================
def find_liquid_line(roi, sensitivity=1.0):
    """
    roi: صورة مقصوصة للكيس فقط (RGB)
    يرجع: رقم الصف (row) اللي فيه خط السائل، من الأعلى
    """
    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # متوسط السطوع لكل صف (سطر أفقي) في الكيس
    row_means = np.mean(blur, axis=1)

    # نعمل smoothing إضافي عشان نتجنب تذبذب بسيط يعطي خط غلط
    kernel_size = max(3, roi.shape[0] // 25)
    if kernel_size % 2 == 0:
        kernel_size += 1
    row_smooth = cv2.GaussianBlur(row_means.reshape(-1, 1), (1, kernel_size), 0).flatten()

    # نحسب التغيّر (gradient) بين كل صف والي بعده
    gradient = np.abs(np.diff(row_smooth))

    # نطبق الحساسية: كل ما زادت الحساسية كل ما قبلنا تغييرات أصغر
    threshold = np.max(gradient) * (0.15 / sensitivity)
    candidates = np.where(gradient > threshold)[0]

    if len(candidates) == 0:
        # ما لقينا خط واضح - نفترض النص كتقدير احتياطي
        return roi.shape[0] // 2

    # نختار أقوى نقطة تغيّر (أكبر فرق بالسطوع) كخط السائل
    line_row = candidates[np.argmax(gradient[candidates])]

    return int(line_row)


# ============================================================
# الدالة الرئيسية
# ============================================================
def detect_iv(img, sensitivity=1.0):
    if img is None:
        return None, "ارفع صورة أولاً"

    box = find_bag_box(img)
    if box is None:
        return img, "لم يتم اكتشاف الكيس - جرب إضاءة أوضح أو صورة أقرب"

    x, y, w, h = box
    roi = img[y:y + h, x:x + w]

    line_row = find_liquid_line(roi, sensitivity)

    # النسبة = المسافة من خط السائل للأسفل ÷ الارتفاع الكلي
    # (السائل يملأ من الأسفل للأعلى، فكل ما كان الخط لفوق كل ما زادت النسبة)
    liquid_ratio = ((h - line_row) / h) * 100
    liquid_ratio = max(0, min(100, liquid_ratio))

    if liquid_ratio > 75:
        status = "FULL"
    elif liquid_ratio > 45:
        status = "HALF"
    elif liquid_ratio > 20:
        status = "QUARTER"
    else:
        status = "EMPTY"

    # -----------------------------
    # رسم النتيجة
    # -----------------------------
    output = img.copy()

    # صندوق الكيس كامل (أخضر)
    cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 3)

    # خط السائل الفعلي (أصفر) - هذا أهم شي يبين وين قاس البرنامج
    abs_line_row = y + line_row
    cv2.line(output, (x, abs_line_row), (x + w, abs_line_row), (255, 255, 0), 3)

    label = f"{status} {liquid_ratio:.1f}%"
    cv2.putText(
        output,
        label,
        (x, max(y - 10, 25)),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 0, 0),
        2,
    )

    result_text = f"""IV Fluid Level: {liquid_ratio:.1f}%
Status: {status}
(الخط الأصفر = مستوى السائل اللي حسبه البرنامج - إذا كان غلط زوّد/قلل الحساسية)"""

    return output, result_text


# ============================================================
# واجهة Gradio
# ============================================================
demo = gr.Interface(
    fn=detect_iv,
    inputs=[
        gr.Image(type="numpy", label="ارفع صورة المحلول"),
        gr.Slider(0.3, 3.0, value=1.0, step=0.1, label="الحساسية (زوّدها لو ما يلقى خط واضح)"),
    ],
    outputs=[
        gr.Image(label="Detection"),
        gr.Textbox(label="النتيجة"),
    ],
    title="Nurse AI - IV Fluid Monitoring v2",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://886926c1eb69af52aa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
# ================================
# IV Fluid Level Evaluation Table
# ضع هذا الكود بعد Gradio مباشرة
# ================================

import pandas as pd
import numpy as np

# نسبة الخطأ المقبولة
# يعني إذا الفرق بين الحقيقي والمتوقع 10% أو أقل يعتبر Pass
tolerance = 10

# ================================
# اكتب بيانات الصور هنا
# Actual = النسبة الحقيقية
# Predicted = النسبة التي أخرجها البرنامج
# ================================

data = [
    {
        "Sample": "Image 1",
        "Actual IV Fluid %": 75.0,
        "Predicted IV Fluid %": 56.6,
        "Predicted Status": "HALF"
    },

    # إذا عندكم صور ثانية، أضفها بنفس الطريقة:
    # {
    #     "Sample": "Image 2",
    #     "Actual IV Fluid %": 50.0,
    #     "Predicted IV Fluid %": 48.5,
    #     "Predicted Status": "HALF"
    # },
]

# ================================
# Create Evaluation Table
# ================================

df = pd.DataFrame(data)

df["Error %"] = abs(df["Actual IV Fluid %"] - df["Predicted IV Fluid %"])

df["Result"] = df["Error %"].apply(
    lambda error: "Pass" if error <= tolerance else "Fail"
)

df = df.round(2)

display(df)

# ================================
# Evaluation Metrics
# ================================

mae = df["Error %"].mean()
rmse = np.sqrt(((df["Actual IV Fluid %"] - df["Predicted IV Fluid %"]) ** 2).mean())
accuracy = (df["Result"] == "Pass").mean() * 100

print("========== Evaluation Summary ==========")
print(f"Mean Absolute Error (MAE): {mae:.2f}%")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}%")
print(f"Accuracy within ±{tolerance}%: {accuracy:.2f}%")

# ================================
# Best / Worst Result
# ================================

best_sample = df.loc[df["Error %"].idxmin()]
worst_sample = df.loc[df["Error %"].idxmax()]

print("\n========== Best Sample ==========")
print(f"Sample: {best_sample['Sample']}")
print(f"Error: {best_sample['Error %']}%")

print("\n========== Worst Sample ==========")
print(f"Sample: {worst_sample['Sample']}")
print(f"Error: {worst_sample['Error %']}%")

# ================================
# Failure Report
# ================================

failure_1_reason = "The predicted IV fluid percentage was different from the actual fluid level."
failure_1_fix = "Improve the model by testing it on more IV fluid images with different fluid levels."

failure_2_reason = "Lighting, camera angle, or unclear fluid line may affect the prediction."
failure_2_fix = "Use clear images with the same camera distance, angle, and lighting."

ready_for_hospital_deployment = False

print("\n========== Failure Report ==========")
print(f"Failure 1: {failure_1_reason}")
print(f"Fix: {failure_1_fix}")

print(f"\nFailure 2: {failure_2_reason}")
print(f"Fix: {failure_2_fix}")

print(f"\nReady for hospital deployment: {ready_for_hospital_deployment}")

,Sample,Actual IV Fluid %,Predicted IV Fluid %,Predicted Status,Error %,Result
0,Image 1,75.0,56.6,HALF,18.4,Fail


========== Evaluation Summary ==========
Mean Absolute Error (MAE): 18.40%
Root Mean Squared Error (RMSE): 18.40%
Accuracy within ±10%: 0.00%

========== Best Sample ==========
Sample: Image 1
Error: 18.4%

========== Worst Sample ==========
Sample: Image 1
Error: 18.4%

========== Failure Report ==========
Failure 1: The predicted IV fluid percentage was different from the actual fluid level.
Fix: Improve the model by testing it on more IV fluid images with different fluid levels.

Failure 2: Lighting, camera angle, or unclear fluid line may affect the prediction.
Fix: Use clear images with the same camera distance, angle, and lighting.

Ready for hospital deployment: False
